<a href="https://colab.research.google.com/github/op1154/MBA-AI/blob/main/Llama3_1_(8B)_Alpaca_FineTunning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Instalação

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Llama-3.1-70B-bnb-4bit",
    "unsloth/Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [15]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

instruction = "Você é um especialista em assuntos da FIAP."
user_input = "O que é a Quantum Commerce?."

inputs = tokenizer(
    [
        alpaca_prompt.format(
            instruction, # Instruction
            user_input, # Input
            "", # Output - leave this blank for generation!
        )
    ], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])

<|begin_of_text|>Abaixo está uma instrução que descreve uma tarefa, combinada com uma entrada que fornece mais contexto. Escreva uma resposta que complete adequadamente o pedido.

### Instruction:
Você é um especialista em assuntos da FIAP.

### Input:
O que é a Quantum Commerce?.

### Response:
A solução de e-commerce desenvolvida por alunos da FIAP em São Paulo como projeto educacional multidisciplinar.<|end_of_text|>


Agora adicionamos adaptadores LoRA para que precisemos atualizar apenas 1 a 10% de todos os parâmetros!

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.7.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


<a name="Data"></a>
### Preparação de Dados
Agora usamos o [conjunto de dados Alpaca](https://huggingface.co/datasets/unsloth/alpaca-cleaned), que é uma versão filtrada de 52K do conjunto original [Alpaca](https://crfm.stanford.edu/2023/03/13/alpaca.html). Você pode substituir essa seção de código pelo seu próprio preparo de dados.

**[NOTA]** Para treinar apenas nas respostas (ignorando a entrada do usuário), leia nossa documentação [aqui](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide/lora-hyperparameters-guide#training-on-completions-only-masking-out-inputs)

**[NOTA]** Lembre-se de adicionar o **EOS_TOKEN** ao output tokenizado! Caso contrário, você terá gerações infinitas!

Se quiser usar o template `llama-3` para datasets ShareGPT, experimente nosso notebook conversacional [aqui](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Conversational.ipynb)

Para completions de texto, como escrita de romances, experimente este [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Mistral_(7B)-Text_Completion.ipynb).

In [6]:
alpaca_prompt = """Abaixo está uma instrução que descreve uma tarefa, combinada com uma entrada que fornece mais contexto. Escreva uma resposta que complete adequadamente o pedido.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Adicionando EOS_TOKEN ao final de cada pergunta e resposta

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Deve ser adicionado o EOS_TOKEN, caso contrario o seu modelo irá trazer dados eternamente!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

from datasets import load_dataset
dataset = load_dataset("json", data_files="/content/my_dataset.json", split = "train")

dataset = dataset.map(formatting_prompts_func, batched = True,)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/28 [00:00<?, ? examples/s]

In [7]:
dataset

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 28
})

<a name="Train"></a>
### Treinamento do modelo
Agora vamos treinar nosso modelo. Fizemos 60 passos para agilizar, mas você pode definir `num_train_epochs=1` para uma execução completa e desativar `max_steps=None`. Também suportamos `DPOTrainer` e `GRPOTrainer` para reinforcement learning!!

In [8]:
# Importa as classes de configuração e treinamento para Supervised Fine-Tuning (SFT) da biblioteca TRL
from trl import SFTConfig, SFTTrainer

# Inicializa o framework de treinamento do modelo
trainer = SFTTrainer(
    model = model,                     # O modelo de linguagem que será treinado
    tokenizer = tokenizer,             # O tokenizador correspondente ao modelo para processar o texto
    train_dataset = dataset,           # O conjunto de dados (dataset) usado para o treinamento
    dataset_text_field = "text",       # O nome da coluna do dataset que contém o texto de treino
    max_seq_length = max_seq_length,   # O tamanho máximo (em tokens) que cada sequência de texto pode ter
    packing = False, # Can make training 5x faster for short sequences. -> Mantém os textos separados (se True, junta textos curtos)

    # Define os hiperparâmetros e configurações técnicas do treinamento
    args = SFTConfig(
        per_device_train_batch_size = 2,  # Quantidade de exemplos processados por vez em cada placa (GPU)
        gradient_accumulation_steps = 4,  # Acumula gradientes por 4 passos antes de atualizar o modelo (simula um batch size maior)
        warmup_steps = 5,                 # Quantidade de passos iniciais onde a taxa de aprendizado cresce gradualmente
        # num_train_epochs = 1, # Set this for 1 full training run. -> Treinar por 1 época completa (passar pelo dataset todo uma vez)
        max_steps = 60,                   # Número total máximo de passos de treinamento (ignora o número de épocas se ativo)
        learning_rate = 2e-4,             # Taxa de aprendizado inicial (velocidade com que o modelo ajusta os pesos)
        logging_steps = 1,                # Frequência (em passos) com que as métricas de perda (loss) serão exibidas na tela
        optim = "adamw_8bit",             # Otimizador em 8-bits para economizar muita memória VRAM da GPU
        weight_decay = 0.001,             # Penalidade aplicada aos pesos do modelo para evitar overfitting (decoreba)
        lr_scheduler_type = "linear",     # Como a taxa de aprendizado diminui ao longo do tempo (redução linear)
        seed = 3407,                      # Semente aleatória para garantir que os resultados sejam reproduzíveis
        output_dir = "outputs",           # Pasta onde os checkpoints e os arquivos finais do modelo serão salvos
        report_to = "none", # Use TrackIO/WandB etc -> Desativa o envio de métricas para ferramentas externas (como Weights & Biases)
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/28 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [9]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
6.705 GB of memory reserved.


In [10]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 28 | Num Epochs = 15 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,2.795200
2,2.892700
3,2.853700
4,2.604300
5,2.223600
6,1.965500
7,1.506700
8,1.144800
9,0.878500
10,0.625200


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


In [11]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

242.1894 seconds used for training.
4.04 minutes used for training.
Peak reserved memory = 6.705 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 46.041 %.
Peak reserved memory for training % of max memory = 0.0 %.


<a name="Inference"></a>
### Inferência
Vamos executar o modelo! Você pode alterar a instrução e a entrada — deixe a saída em branco!

In [12]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Você é um agente que fala sobre a empresa Quantum Commerce", # instruction
        "O que é a Quantum Commerce?", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

['<|begin_of_text|>Abaixo está uma instrução que descreve uma tarefa, combinada com uma entrada que fornece mais contexto. Escreva uma resposta que complete adequadamente o pedido.\n\n### Instruction:\nVocê é um agente que fala sobre a empresa Quantum Commerce\n\n### Input:\nO que é a Quantum Commerce?\n\n### Response:\nA Quantum Commerce é uma empresa fictícia de e-commerce criada por alunos da FIAP em São Paulo como projeto acadêmico.<|end_of_text|>']

Você também pode usar um `TextStreamer` para inferência contínua — assim você verá a geração token a token, em vez de esperar o resultado completo!

In [16]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Você é um agente que fala sobre a empresa Quantum Commerce", # instruction
        "O que é a Quantum Commerce?", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<|begin_of_text|>Abaixo está uma instrução que descreve uma tarefa, combinada com uma entrada que fornece mais contexto. Escreva uma resposta que complete adequadamente o pedido.

### Instruction:
Você é um agente que fala sobre a empresa Quantum Commerce

### Input:
O que é a Quantum Commerce?

### Response:
A Quantum Commerce é uma empresa fictícia de e-commerce criada por alunos da FIAP em São Paulo como projeto acadêmico.<|end_of_text|>


<a name="Save"></a>
### Salvando e carregando modelos fine-tuned
Para salvar o modelo final como adaptadores LoRA, use `push_to_hub` do Hugging Face para salvar online ou `save_pretrained` para salvar localmente.

**[NOTA]** Isso salva SOMENTE os adaptadores LoRA, e não o modelo completo. Para salvar em 16bit ou GGUF, role para baixo!

In [17]:
model.save_pretrained("llama_lora")  # Local saving
tokenizer.save_pretrained("llama_lora")
# model.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving

('llama_lora/tokenizer_config.json',
 'llama_lora/special_tokens_map.json',
 'llama_lora/tokenizer.json')

Agora, se quiser carregar os adaptadores LoRA que acabamos de salvar para inferência, mude `False` para `True`:

In [18]:
if True:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "llama_lora", # Seu modelo que foi usado para treinar
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # 2x mais rápido para fazer a inferencia

# alpaca_prompt = You MUST copy from above!

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Qual empresa acadêmica foi criada pelos alunos da FIAP?", # instruction
        "", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

==((====))==  Unsloth 2026.7.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
<|begin_of_text|>Abaixo está uma instrução que descreve uma tarefa, combinada com uma entrada que fornece mais contexto. Escreva uma resposta que complete adequadamente o pedido.

### Instruction:
Qual empresa acadêmica foi criada pelos alunos da FIAP?

### Input:


### Response:
A FIAP é uma empresa fictícia de ensino superior desenvolvida por alunos da FIAP em São Paulo como projeto acadêmico.<|end_of_text|>


Você também pode usar `AutoPeftModelForCausalLM` do Hugging Face. Use isto apenas se não tiver `unsloth` instalado. Pode ser muito lento, já que o download de modelos em `4bit` não é suportado, e a inferência do Unsloth é **2x mais rápida**.

In [19]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "llama_lora", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("llama_lora")

### Salvando em float16 para VLLM

Também suportamos salvar diretamente em `float16`. Selecione `merged_16bit` para float16 ou `merged_4bit` para int4. Também permitimos adaptadores `lora` como fallback. Use `push_to_hub_merged` para enviar para sua conta no Hugging Face! Acesse https://huggingface.co/settings/tokens para seus tokens pessoais. Veja [nossa documentação](https://unsloth.ai/docs/basics/inference-and-deployment) para mais opções de deploy.

In [21]:
# Cria para 16bit
if False: model.save_pretrained_merged("llama_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/llama_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Bria para 4bit
if True: model.save_pretrained_merged("llama_finetune_4bit", tokenizer, save_method = "merged_4bit_forced",) # Alterado para 'merged_4bit_forced'
if False: model.push_to_hub_merged("HF_USERNAME/llama_finetune_4bit", tokenizer, save_method = "merged_4bit_forced", token = "YOUR_HF_TOKEN") # Alterado para 'merged_4bit_forced'

# Para LoRA adapters
if True:
    model.save_pretrained("llama_lora")
    tokenizer.save_pretrained("llama_lora")
if False:
    model.push_to_hub("HF_USERNAME/llama_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/llama_lora", token = "YOUR_HF_TOKEN")

config.json:   0%|          | 0.00/942 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Unsloth: Merging LoRA weights into 4bit model...


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Unsloth: Merging finished.
Unsloth: Found skipped modules: ['model.layers.1.mlp.gate_proj', 'model.layers.1.mlp.up_proj', 'model.layers.1.mlp.down_proj', 'lm_head']. Updating config.
Unsloth: Saving merged 4bit model to llama_finetune_4bit...
Unsloth: Merged 4bit model saved.
Unsloth: Merged 4bit model process completed.


HfHubHTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/api/repos/create (Request ID: Root=1-6a4db23a-02c9cec310fe9bdc7129e6a6;03ef8c2e-ef0b-4556-ac4c-0dd1d114d405)

Invalid username or password.

### Conversão para GGUF / llama.cpp
Para salvar em `GGUF` / `llama.cpp`, agora suportamos nativamente! Clonamos `llama.cpp` e o padrão é salvar em `q8_0`. Permitimos todos os métodos como `q4_k_m`. Use `save_pretrained_gguf` para salvar localmente e `push_to_hub_gguf` para enviar ao HF.

Alguns métodos de quantização suportados (lista completa na nossa [página de docs](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Conversão rápida. Alto uso de recursos, mas geralmente aceitável.
* `q4_k_m` - Recomendado. Usa Q6_K para metade dos tensores attention.wv e feed_forward.w2, caso contrário Q4_K.
* `q5_k_m` - Recomendado. Usa Q6_K para metade dos tensores attention.wv e feed_forward.w2, caso contrário Q5_K.

[**NOVO**] Para fine-tune e export automático para Ollama, experimente nosso [notebook Ollama](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Salve para 8bit Q8_0
if False: model.save_pretrained_gguf("llama_finetune", tokenizer,)
# Lembre-se de criar um token no https://huggingface.co/settings/tokens!
# Coloque o seu username!
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Salve para 16bit GGUF
if False: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Salve para q4_k_m GGUF
if True: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Salve para multiplos GGUF
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/llama_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN",
    )

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

In [ ]:
# Salve no seu GoogleDrive para posteriormente poder baixá-lo
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title Aula Parte 2 sobre os modelos de avaliação de LLM
import evaluate
import pandas as pd
from transformers import pipeline

# ==========================================
# 1. FUNÇÃO AUXILIAR DE AVALIAÇÃO
# ==========================================
def avaliar_modelo(model, tokenizer, dataset):
    """Gera respostas com o modelo e calcula ROUGE e BERTScore."""
    # Cria um pipeline de geração de texto otimizado
    gerador = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=50,
        # device=0  # Use -1 se estiver rodando apenas em CPU (Removido para evitar conflito com accelerate)
    )

    # Extrai perguntas e referências do dataset de teste
    perguntas = [item["prompt"] for item in dataset_teste]
    referencias = [item["chosen"] for item in dataset_teste]

    # Gera as respostas usando o modelo atual
    respostas_modelo = []
    for p in perguntas:
        resultado = gerador(p)
        # Extrai apenas o texto gerado após o prompt do usuário
        texto_gerado = resultado[0]["generated_text"][len(p):].strip()
        respostas_modelo.append(texto_gerado)

    # Carrega as métricas do Hugging Face
    rouge_metric = evaluate.load("rouge")
    bert_metric = evaluate.load("bertscore")

    # Calcula as pontuações
    rouge_res = rouge_metric.compute(predictions=respostas_modelo, references=referencias, use_stemmer=True)
    bert_res = bert_metric.compute(predictions=respostas_modelo, references=referencias, lang="pt")

    # Retorna o dicionário consolidado com as médias
    return {
        "ROUGE-1": round(rouge_res["rouge1"], 4),
        "ROUGE-2": round(rouge_res["rouge2"], 4),
        "ROUGE-L": round(rouge_res["rougeL"], 4),
        "BERTScore-F1": round(sum(bert_res["f1"]) / len(bert_res["f1"]), 4)
    }

# ==========================================
# 2. DEFINIÇÃO DO DATASET DE TESTE (Exemplo)
# ==========================================
# Substitua pelo seu dataset real de validação/teste separado do treino
dataset_teste = [
    {"prompt": "Pergunta: Quem desenvolveu a Quantum Commerce? Resposta:", "chosen": "A solução foi idealizada e desenvolvida por alunos da FIAP como um projeto educacional multidisciplinar."},
    {"prompt": "Pergunta: Qual é a missão da Quantum Commerce?? Resposta:", "chosen": "Oferecer uma experiência de compra digital inteligente, segura e personalizada.."}
]

# ==========================================
# 3. AVALIAÇÃO ANTES DO TREINAMENTO
# ==========================================
print("\n[1/3] Avaliando o modelo base (Antes do treino)...")
metricas_antes = avaliar_modelo(trainer.model, trainer.tokenizer, dataset_teste)

# ==========================================
# 4. EXECUÇÃO DO TREINAMENTO (60 PASSOS)
# ==========================================
print("\n[2/3] Iniciando o Fine-Tuning...")
#trainer.train()

# Salva o resultado final do treino
#trainer.model.save_pretrained("modelo_final_sft")
#trainer.tokenizer.save_pretrained("modelo_final_sft")

# ==========================================
# 5. AVALIAÇÃO DEPOIS DO TREINAMENTO
# ==========================================
print("\n[3/3] Avaliando o modelo ajustado (Depois do treino)...")
# Usamos o modelo atualizado que está dentro do próprio trainer
metricas_depois = avaliar_modelo(trainer.model, trainer.tokenizer, dataset_teste)

# ==========================================
# 6. EXIBIÇÃO DA TABELA COMPARATIVA FINAL COM PERCENTUAL DE GANHO
# ==========================================
# 1. Cria o DataFrame inicial com as métricas de Antes e Depois
df_comparativo = pd.DataFrame([metricas_antes, metricas_depois], index=["Antes (Base)", "Depois (Fine-Tuning)"])

# 2. Calcula a variação percentual: ((Depois - Antes) / Antes) * 100
# Usamos .loc para garantir o cálculo correto baseado nos índices textuais
ganho_percentual = ((df_comparativo.loc["Depois (Fine-Tuning)"] - df_comparativo.loc["Antes (Base)"]) / df_comparativo.loc["Antes (Base)"]) * 100

# 3. Adiciona a linha de ganho formatada com o símbolo de '%'
df_comparativo.loc["Ganho (%)"] = ganho_percentual.map("{:+.2f}%".format)

print("\n========================================================")
print("          TABELA COMPARATIVA DE EVOLUÇÃO                ")
print("========================================================")
print(df_comparativo.to_string())
print("========================================================")